
# mT5 small — Baseline TF Training (Without Wuxia Domain)

**TFG Anonymous – Baseline NMT (mt5-small)**  
This notebook  evaluates to model **MarianMT** (`google/mt5-small`)without train in dominio using to dataset of **wuxia** (Chinese->English) already preparado in format `datasets` (HF).





## 1) Environment of execution and installation of dependencies

In [ ]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Name of the GPU:", torch.cuda.get_device_name(0))



> **Requirements of the dataset**: directory HF Datasets with *splits* `train`, `validation`, `test` and columns `zh` (Chinese) and `en` (English):  
> `processed_data/wuxia_zh_en_clean/`


In [ ]:
# Configuration of carpetas for entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent.parent.parent.parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Structure created (if not existed):")
for p in ["evaluation", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Score: the dataset must existir in: CORPUS/proccesed data/wuxia_zh_en_clean


## 2) Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Paths (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   
    output_dir: Path   = BASE_DIR / "models" / "pretrain_mt5_small"             
    ckpt_dir: Path     = BASE_DIR / "checkpoints"
    training_dir: Path = BASE_DIR / "training"
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    translate_file: Path =   "pre_mt5.txt"
    results_file: Path = "pre_results.txt"
    # Columns of the dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Model 
    model_ckpt: str = "google/mt5-small"

    # Prefix of instruction for T5/mT5 (for avoid <extra_id_0>)
    use_instruction_prefix: bool = True
    translation_prefix: str = "translate Chinese to English: "

    # Training
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3
    fraction: float = 1

cfg = Config()
print(cfg)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas for reproducibilidad
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # For reproducibilidad estricta (ligera penalty of rendimiento)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas and backend configured.")


## 4) Load dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"The dataset was not found at: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columns
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' must contener columns '{src_col}' y '{tgt_col}'. Columns: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

#  tests quick
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fraction={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


## 4) Load tokenizador and model 

In [ ]:

from transformers import MT5Tokenizer, MT5ForConditionalGeneration

# Tokenizer
tokenizer = MT5Tokenizer.from_pretrained(cfg.model_ckpt)

# Model
model = MT5ForConditionalGeneration.from_pretrained(cfg.model_ckpt)

#  Salvaguardas of tokens special
# Pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Decoder start token
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

# EOS token
if model.config.eos_token_id is None:
    model.config.eos_token_id = tokenizer.eos_token_id

# Avoid warnings with gradient checkpointing
model.config.use_cache = False

# Enviar a dispositivo
model.to(device)

# Info
print("Model:", cfg.model_ckpt)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)
print("decoder_start_token_id:", model.config.decoder_start_token_id)


## 5) Preprocesamiento and tokenization

In [ ]:

max_source_length = cfg.max_source_length
max_target_length = cfg.max_target_length

def preprocess_function(examples):
    #  Source (ZH) with prefix of instruction for mT5
    if cfg.use_instruction_prefix:
        src_texts = [cfg.translation_prefix + s.strip() for s in examples[cfg.src_col]]
    else:
        src_texts = [s.strip() for s in examples[cfg.src_col]]

    #  Target (IN)
    tgt_texts = [t.strip() for t in examples[cfg.tgt_col]]

    #  Tokenization: 
    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_length,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=tgt_texts,           # for read labels bien
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed) if "test" in tokenized_datasets else val_ds

print("Example tokenized:", {k: type(v) for k,v in train_ds[0].items()})


## 6) Evaluation 

In [ ]:

from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import json
import time

start = time.time()
# Descargar recursos of NLTK (for METEOR)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

#  Parameters 
EVAL_MAX_SAMPLES = 1000        # None = all the split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)

#  Comprobaciones 
assert 'model' in globals(), "Was not found `model`. Loads the model before."
assert 'tokenizer' in globals(), "Was not found `tokenizer`. Load it before."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."



# Ensure pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# mT5: ensure decoder_start_token_id
if getattr(model.config, 'decoder_start_token_id', None) is None and tokenizer.pad_token_id is not None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id


#  Seleccionar split 
eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No there are examples for evaluar."

def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]

#  mT5: ensure prefix of instruction in the input 
if hasattr(cfg, 'use_instruction_prefix') and cfg.use_instruction_prefix:
    pref = getattr(cfg, 'translation_prefix', 'translate Chinese to English: ')
    def _ensure_pref(s):
        return s if s.startswith(pref) else (pref + s)
    src_texts = [_ensure_pref(s) for s in src_texts]

#  Generation by batches 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

#  Metrics 
bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

chrf_metric = CHRF(word_order=2)
chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

ter_metric = TER()
ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

def compute_rougeL_f1(hyp_list, ref_list):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    f1s = []
    for h, r in zip(hyp_list, ref_list):
        scores = scorer.score(r, h)
        f1s.append(scores['rougeL'].fmeasure * 100.0)
    return float(np.mean(f1s))

rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

def compute_meteor(hyp_list, ref_list):
    scores = []
    for h, r in zip(hyp_list, ref_list):
        hyp_tok = wordpunct_tokenize(h)
        ref_tok = wordpunct_tokenize(r)
        scores.append(meteor_score([ref_tok], hyp_tok))
    return float(np.mean(scores)) * 100.0

meteor_avg = compute_meteor(preds, ref_texts)

end_time = time.time()

results = {
    "model" : cfg.model_ckpt,
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    "sacrebleu": round(bleu_corpus, 4),
    "chrf2": round(chrf_corpus, 4),
    "ter": round(ter_corpus, 4),
    "rougeL_f1": round(rougeL_f1, 4),
    "meteor": round(meteor_avg, 4), 
    "execution_time": round(end_time - start, 2)
}

os.makedirs(cfg.evaluation_dir, exist_ok=True)

res_file = os.path.join(cfg.evaluation_dir, cfg.results_file)

with open(res_file, "a", encoding="utf-8") as f:
    f.write("\n")
    f.write(json.dumps(results, ensure_ascii=False, indent=4))

print(results)



In [ ]:
print(res_file)

## 12) Sample cualitativa (n examples aleatorios)

In [ ]:
import random
import torch

# Mostrar predicciones aleatorias 
n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decode text source and reference from dataset tokenized
    zh_in = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # If the input already contains the prefix, it retiramos only for print "ZH:"
    if cfg.use_instruction_prefix and zh_in.startswith(cfg.translation_prefix):
        zh_print = zh_in[len(cfg.translation_prefix):]
    else:
        zh_print = zh_in

    # Prepare input for generate (siempre with prefix)
    zh_for_gen = cfg.translation_prefix + zh_print if cfg.use_instruction_prefix else zh_print
    inputs = tokenizer([zh_for_gen], return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_source_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh_print)
    print("IN (ref):", en_ref)
    print("IN (pred):", en_pred)


In [ ]:

os.makedirs(cfg.translate_dir, exist_ok=True)
translate_path = os.path.join(cfg.translate_dir, cfg.translate_file)

with open(translate_path, "w", encoding="utf-8") as f:
    for i in range(len(eval_raw) // 2):
        zh_in = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
        en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

        # If the input already contains the prefix, it retiramos only for print "ZH:"
        if cfg.use_instruction_prefix and zh_in.startswith(cfg.translation_prefix):
            zh_print = zh_in[len(cfg.translation_prefix):]
        else:
            zh_print = zh_in

        # Prepare input for generate siempre with prefix (avoid <id_0> or salidas vacias)
        zh_for_gen = cfg.translation_prefix + zh_print if cfg.use_instruction_prefix else zh_print
        inputs = tokenizer([zh_for_gen], return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_source_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_length=cfg.max_target_length,
                num_beams=4,
                early_stopping=True
            )

        en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

        # Save in the file
        f.write("="*80 + "\n")
        f.write("ZH: " + zh_in + "\n")
        f.write("IN (ref): " + en_ref + "\n")
        f.write("IN (pred): " + en_pred + "\n\n")
